RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/home/zeropoint/Projects/CUSTOMRAG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
###
def process_all_pdfs(pdf_directory):
    """Process all pdfs directly"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    # Find all pdfs
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name} ")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = "pdf"
            
            all_documents.extend(documents)
        
        except Exception as exception:
            print(f"Error {str(exception)}")
        
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_documents = process_all_pdfs("../data")

Found 3 PDF files to process

 Processing: attention_mechanisms_basic_information.pdf 

 Processing: embeddings_basic_information.pdf 

 Processing: llm_basic_information.pdf 
Total documents loaded: 3


In [3]:
###Textplitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG Performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks = split_documents(all_documents)

Split 3 documents into 3 chunks

 Example chunk:
Content Basic Information About Attention Mechanisms
Attention mechanisms are techniques used in machine learning models to decide which parts
of the input are most important for a given task. Instead of trea
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-11T07:13:12+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-11T07:13:12+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf_files/attention_mechanisms_basic_information.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention_mechanisms_basic_information.pdf', 'file_type': 'pdf'}


Embedding and vectorStoreDB

In [4]:
import uuid
import chromadb
import numpy as np
from chromadb.config import Settings
from typing import List, Dict, Tuple, Any
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [5]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """
        Load the SentenceTransformer model
        """
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.get_embeddings_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def get_embeddings_dimension(self):
        """
        Get the embedding dimension of the model
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        return self.model.get_sentence_embedding_dimension()

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """"
        Generate embeddings for a list of Texts
        Args:
            text: List of text strings to embed
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generate embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embeddings with shape: {embeddings.shape}")
        return embeddings
    
## Initializing the embedding Manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3872.65it/s]


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_1377162/1781214019.py:35: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


### Vector Store

In [6]:
class VectorStore:
    """
        Manages document embedding in a ChromaDB Vector 
    """

    def __init__(self, collection_name:str = "pdf_documents", persist_directory:str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the Chromadb collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        

    def _initialize_store(self):
        """
            Initialize ChromaDB client and collection
        """

        try:
            #Creating persisten ChromaDb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            pass
        except Exception as exception:
            print(f"Error Initializing Vector Store..")
            raise

    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adding documents and their embedding to the vector store
        
        Args:
            documents: List of langchain documents
            embeddings: Corresponding embedding for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings...")
        
        print(f"Adding {len(documents)} documents to vector store..")

        ids, metadatas , documents_text, embeddings_list = [], [] ,[] ,[]

        for index, (document, embedding) in enumerate(zip(documents, embeddings)):
            document_id = f"doc_{uuid.uuid4().hex[:8]}_{index}"
            ids.append(document_id)

            #prepare metadata
            metadata = dict(document.metadata)
            metadata['document_index'] = index
            metadata['content_length'] = len(document.page_content)
            metadatas.append(metadata)

            documents_text.append(document.page_content)
            embeddings_list.append(embedding.tolist())

            #Add to collection

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Sucessfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as exception:
            print(f"Error adding documents to vector store {exception}")
            raise
        

vectorstore = VectorStore()
vectorstore

Vector Store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [8]:
from langchain_core.documents import Document
### Convert the text to embedding
documents = chunks

### Generate the embeddings

embeddings = embedding_manager.generate_embeddings(
    texts=[document.page_content for document in documents])

##store in the vector database

vectorstore.add_documents(documents=documents, embeddings=embeddings)

Generate embeddings for 3 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.08it/s]

Generated Embeddings with shape: (3, 384)
Adding 3 documents to vector store..
page_content='Basic Information About Attention Mechanisms
Attention mechanisms are techniques used in machine learning models to decide which parts
of the input are most important for a given task. Instead of treating every word or token
equally, attention allows the model to focus more strongly on relevant information.
In language models, attention helps connect words across a sentence or document. For
example, it can help a model understand which noun a pronoun refers to, or which earlier
phrase gives meaning to a later statement.
The Transformer architecture uses self-attention as a key component. This design made it
possible to train powerful models that process language more efficiently and capture
long-range relationships in text.' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-11T07:13:12+00:00', 'author': '(anonymous)', 'keywords': 